In [2]:
pip install pandas numpy scikit-learn streamlit


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
pip install --upgrade pip


  Using cached pip-26.0.1-py3-none-any.whl.metadata (4.7 kB)
Using cached pip-26.0.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 25.3
    Uninstalling pip-25.3:
      Successfully uninstalled pip-25.3
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [5]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


C:\Users\bekka\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [6]:
pip install --upgrade bottleneck

  Attempting uninstall: bottleneck
    Found existing installation: Bottleneck 1.3.5
    Uninstalling Bottleneck-1.3.5:
      Successfully uninstalled Bottleneck-1.3.5
Note: you may need to restart the kernel to use updated packages.


In [8]:
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import gc
from IPython.display import Markdown
import pickle
from scipy.sparse import csr_matrix
np.random.seed(3213)

In [9]:
ratings = pd.read_csv('/kaggle/input/movie-recommendation-system/ratings.csv', engine="pyarrow")
ratings.head()

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/movie-recommendation-system/ratings.csv'

In [10]:
import pandas as pd

In [14]:
pip install pyarrow

Note: you may need to restart the kernel to use updated packages.


In [17]:
ratings = pd.read_csv("dataset.ratings.csv")
ratings.head()

,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510


In [19]:
ratings = pd.read_csv("dataset.ratings.csv", engine="pyarrow")
ratings.head()

,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510


In [20]:
ratings.shape

(11, 4)

In [21]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   userId     11 non-null     int64  
 1   movieId    11 non-null     int64  
 2   rating     11 non-null     float64
 3   timestamp  11 non-null     int64  
dtypes: float64(1), int64(3)
memory usage: 484.0 bytes


In [22]:
ratings.describe()

,userId,movieId,rating,timestamp
count,11.0,11.000000,11.000000,1.100000e+01
mean,1.0,881.818182,4.136364,1.147872e+09
std,0.0,412.070096,0.710314,5.116920e+03
min,1.0,296.000000,3.500000,1.147868e+09
25%,1.0,486.000000,3.500000,1.147869e+09
50%,1.0,1088.000000,4.000000,1.147869e+09
75%,1.0,1227.000000,5.000000,1.147878e+09
max,1.0,1260.000000,5.000000,1.147880e+09


In [23]:
ratings.columns

Index(['userId', 'movieId', 'rating', 'timestamp'], dtype='object')

In [24]:
ratings['userId'] = ratings['userId'].astype("int32")
ratings['movieId'] = ratings['movieId'].astype("int32")
ratings['rating'] = ratings['rating'].astype("float32")

# Essa coluna sem significância relevante também pode comprometer computacionalmente nossas análises por isso iremos removê-la
ratings.drop('timestamp', axis=1, inplace=True)
print(ratings.dtypes)

userId       int32
movieId      int32
rating     float32
dtype: object


In [25]:
gc.collect()
df_before = len(ratings)
user_counts = ratings['userId'].value_counts()
movie_counts = ratings['movieId'].value_counts()
filtered_ratings = ratings[
    ratings['userId'].isin(user_counts[user_counts >= 150].index) &
    ratings['movieId'].isin(movie_counts[movie_counts >= 250].index)
]
df_after = len(filtered_ratings)
print("Tamanho do Dataset antes da filtragem: {:d}. Tamanho após a filtragem: {:d}. Redução de {:d} registros".
     format(df_before, df_after, (df_before - df_after)))

Tamanho do Dataset antes da filtragem: 11. Tamanho após a filtragem: 0. Redução de 11 registros


In [29]:
ratings = pd.read_csv("dataset.ratings.csv")
ratings.head()

,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510


In [30]:
filtered_ratings = ratings[ratings["rating"] > 3]

In [31]:
filtered_ratings.head(5)

,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510


In [32]:
filtered_ratings.sample(5)

,userId,movieId,rating,timestamp
8,1,1237,5.0,1147868839
6,1,1175,3.5,1147868826
10,1,1260,3.5,1147877857
7,1,1217,3.5,1147878326
3,1,665,5.0,1147878820


In [33]:
gc.collect()
user_movie_matrix = filtered_ratings.pivot_table(index='userId',
                                                 aggfunc='mean', columns='movieId', values='rating').fillna(0)
gc.collect()
sparse_matrix = csr_matrix(user_movie_matrix)
gc.collect()
sparse_similarity = cosine_similarity(sparse_matrix, dense_output=False)

In [34]:
sparse_similarity.setdiag(0)
sparse_similarity.eliminate_zeros()

In [35]:
rows, cols, sims = [], [], []

for i in range(sparse_similarity.shape[0]):
    gc.collect()
    row = sparse_similarity.getrow(i)
    if row.nnz == 0:
        continue
    top_indices = row.data.argsort()[::-1][:3]  # top-3 maiores
    top_cols = row.indices[top_indices]
    top_sims = row.data[top_indices]
    
    rows.extend([i]*len(top_cols))
    cols.extend(top_cols)
    sims.extend(top_sims)

df_top3 = pd.DataFrame({
    "userId": rows,
    "similarUserId": cols,
      "similarity": sims
})

In [36]:
try:
    del user_movie_matrix, sparse_matrix
except:
    print("Objetos não encontrados no ambiente")


In [40]:
df_top3.loc[df_top3['userId'].isin([12, 13])]

,userId,similarUserId,similarity


In [41]:
df_top3['userid']

KeyError: 'userid'

In [43]:
import pandas as pd

ratings = pd.read_csv("dataset.ratings.csv")

df_top3 = ratings.head(100)

df_top3.loc[df_top3['userId'].isin([12,13])]

,userId,movieId,rating,timestamp


In [44]:
ratings = pd.read_csv("dataset.ratings.csv")


In [46]:
import pandas as pd

# read dataset
ratings = pd.read_csv("dataset.ratings.csv")

# take first 100 rows
df_top3 = ratings.head(100)

# filter users 12 and 13
result = df_top3.loc[df_top3['userId'].isin([12, 13])]

print(result)

Empty DataFrame
Columns: [userId, movieId, rating, timestamp]
Index: []


In [47]:
ratings.head()
ratings.columns
ratings.shape

(11, 4)

In [50]:
import pandas as pd
import gc

# Load dataset
ratings = pd.read_csv("dataset.ratings.csv")

# Take first 100 rows
df_top3 = ratings.head(100).copy()

# Filter ratings >= 4
filtered_ratings = ratings[ratings["rating"] >= 4]

# Create similarUserId column
df_top3.loc[:, "similarUserId"] = df_top3["userId"]

# Free unused memory
gc.collect()

# Get unique similar users
similar_users = df_top3["similarUserId"].unique()

# Filter ratings for similar users
similar_user_ratings = filtered_ratings[
    filtered_ratings["userId"].isin(similar_users)
]

# Sort by userId and rating
result = similar_user_ratings.sort_values(
    by=["userId", "rating"],
    ascending=[True, False]
).head(3)

# Display result
print("Top similar user ratings:\n")
print(result)

Top similar user ratings:

   userId  movieId  rating   timestamp
0       1      296     5.0  1147880044
2       1      307     5.0  1147868828
3       1      665     5.0  1147878820


In [51]:
sorted_ratings = similar_user_ratings.sort_values(by=["userId", "rating"], ascending=[True, False])

# Agrupa por usuário e pega os 3 primeiros de cada grupo
top3_by_user = sorted_ratings.groupby("userId").head(3).reset_index(drop=True)

# Exibe o resultado
print(top3_by_user)

   userId  movieId  rating   timestamp
0       1      296     5.0  1147880044
1       1      307     5.0  1147868828
2       1      665     5.0  1147878820


In [62]:
import pandas as pd

# Load datasets
ratings = pd.read_csv("dataset.ratings.csv")
movies = pd.read_csv("dataset.movie.csv")

# Example: top 3 ratings per user
top3_by_user = ratings.sort_values(["userId", "rating"], ascending=[True, False]) \
                      .groupby("userId") \
                      .head(3)

# Filter specific users (12 and 13)
filtered = top3_by_user[top3_by_user["userId"].isin([12, 13])]

# Get movie IDs
movie_ids = filtered["movieId"].unique()

# Get recommended movie titles
recommended_titles = movies[movies["movieId"].isin(movie_ids)]

# Show result
print(recommended_titles)

Empty DataFrame
Columns: [movieId, title, genres]
Index: []


In [63]:
top3_by_user.to_parquet("top3_by_user.parquet", compression="snappy")


In [64]:
df_top3.to_parquet("user_similarity_top3.parquet", compression="snappy")